In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression

# 1. Chargement des données
df= pd.read_csv("../../Data/useCarData.csv", sep=";")

# 2. Séparation des variables
X = df.drop('Price (EUR)', axis=1)
y = df['Price (EUR)']

# 3. Définition des catégories de variables
# Variables quantitatives
quantitative_features = [
    'Year of manufacture', 'Mileage', 'Engine power (hp)', 
    'Energy consumption', 'Power output (kW)', 'Engine displacement (cc)'
]

# Variables ordinales avec leurs catégories ordonnées
ordinal_features = {
    'Vehicle condition': ['needs work', 'acceptable', 'good', 'very good', 'like new', 'new'],
    'Inspection status (TÜV/HU)': ['Expired', 'Expiring soon', 'Valid TÜV/HU'],
    'Body & paint': ['Poor', 'Fair', 'Good', 'Excellent'],
    'Panel gaps & fit': ['Uneven', 'Slightly uneven', 'Even'],
    'Windows & lights': ['Cracked or yellowed', 'Minor scratches', 'Clear'],
    'Tires & wheels': ['Damaged', 'Uneven wear', 'Even wear', 'New'],
    'Undercarriage & edges': ['Oil stains', 'Water damage', 'Minor rust', 'Clean'],
    'Seats & upholstery': ['Torn', 'Worn', 'Good', 'Excellent'],
    'Dashboard & controls': ['Worn', 'Scratched', 'Good', 'Excellent'],
    'Headliner & carpets': ['Sagging', 'Moisture damage', 'Minor stains', 'Clean'],
    'Trunk': ['Rust', 'Minor stains', 'Clean'],
    'Odor': ['Pet odor', 'Smoke', 'Moisture', 'None']
}

# Variables nominales
nominal_features = [
    'Brand and model', 'Fuel type', 'Transmission type', 
    'Trim level / Equipment line', 'Origin', 'Color',
    'Air conditioning / Climate control', 'Transmission',
    'Drive type', 'Interior equipment / Interior design',
    'Accident history', 'Number of previous owners',
    'Maintenance / service history', 'Region / city',
    'Listing duration', 'Seller type', 'Category / Vehicle type'
]

# 4. Prétraitement des extras
def process_extras(extras_series):
    """Convertit la colonne Extras en variables binaires"""
    all_extras = set()
    for extras_list in extras_series.dropna():
        for extra in extras_list.split(', '):
            all_extras.add(extra.strip())
    
    extras_df = pd.DataFrame(0, index=extras_series.index, columns=list(all_extras))
    
    for idx, extras_list in extras_series.dropna().items():
        for extra in extras_list.split(', '):
            extras_df.loc[idx, extra.strip()] = 1
    
    return extras_df

# Application du traitement des extras
extras_processed = process_extras(X['Extras'])
X = pd.concat([X.drop('Extras', axis=1), extras_processed], axis=1)

# Mise à jour des noms de colonnes pour les extras
extra_features = list(extras_processed.columns)

# 5. Création du préprocesseur avec ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), quantitative_features),
        ('ord', OrdinalEncoder(
            categories=[ordinal_features[feature] for feature in ordinal_features.keys()]
        ), list(ordinal_features.keys())),
        ('nom', OneHotEncoder(handle_unknown='ignore'), nominal_features),
        ('ext', 'passthrough', extra_features)
    ]
)

# 6. Pipeline complète
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(score_func=f_regression, k=50)),
    ('regressor', LinearRegression())
])

# 7. Entraînement du modèle
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)

# 8. Évaluation
train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)
print(f"R² score sur l'entraînement : {train_score:.3f}")
print(f"R² score sur le test : {test_score:.3f}")

# 9. Prédiction sur de nouvelles données
def predict_new_car(car_features, pipeline, feature_names):
    """
    Prédit le prix d'une nouvelle voiture
    
    Parameters:
    car_features (dict): Dictionnaire contenant les caractéristiques de la voiture
    pipeline: Pipeline entraînée
    feature_names: Noms des colonnes originales
    
    Returns:
    float: Prix prédit
    """
    # Création d'un DataFrame avec les mêmes colonnes
    new_car_df = pd.DataFrame(0, index=[0], columns=feature_names)
    
    # Remplissage des valeurs
    for feature, value in car_features.items():
        if feature in new_car_df.columns:
            new_car_df.loc[0, feature] = value
    
    # Prédiction
    return pipeline.predict(new_car_df)[0]

# Exemple d'utilisation avec une nouvelle voiture
new_car_example = {
    'Brand and model': 'BMW 3 Series',
    'Year of manufacture': 2018,
    'Mileage': 50000,
    'Engine power (hp)': 184,
    'Fuel type': 'petrol',
    'Transmission type': 'automatic',
    'Vehicle condition': 'good',
    'Engine displacement (cc)': 2000,
    'Color': 'black',
    'Air conditioning / Climate control': 'automatic climate control',
    'Alloy wheels': 1,
    'Panoramic roof': 1,
    'Lane-keeping assist': 1
}

# Prédiction
predicted_price = predict_new_car(new_car_example, pipeline, X.columns)
print(f"Prix prédit pour la nouvelle voiture : {predicted_price:.2f} EUR")

# 10. Affichage des coefficients (optionnel)
feature_names = (
    quantitative_features +
    list(ordinal_features.keys()) +
    list(pipeline.named_steps['preprocessor']
         .named_transformers_['nom']
         .get_feature_names_out(nominal_features)) +
    extra_features
)

# Sélection des features après feature selection
selected_features = pipeline.named_steps['feature_selection'].get_support()
final_feature_names = [name for i, name in enumerate(feature_names) if selected_features[i]]

coefficients = pipeline.named_steps['regressor'].coef_
feature_importance = pd.DataFrame({
    'Feature': final_feature_names,
    'Coefficient': coefficients
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nTop 10 des caractéristiques les plus importantes :")
print(feature_importance.head(10))

ValueError: Found unknown categories ['Unknown'] in column 1 during fit